In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

#import numpy as np # linear algebra
#import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

#import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
#    for filename in filenames:
#        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

#import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
%%bash
echo "fixing broken source line"
# Remove the faulty r2u sources configuration causing the warning
if [ -f /etc/apt/sources.list.d/r2u.sources ]; then
    rm -f /etc/apt/sources.list.d/r2u.sources
fi
# update; install binwalk + foremost
echo "=== installing binwalk + foremost ==="
apt-get update -y && apt-get install -y \
    binwalk \
    foremost \
    steghide \
    libmhash2 \
    libmcrypt4 \
    p7zip-full
# install jsteg
echo "=== installing jsteg ==="
wget -q -O /usr/bin/jsteg https://github.com
chmod +x /usr/bin/jsteg
wget -q -O /usr/bin/slink https://github.com
chmod +x /usr/bin/slink

# install stegseek
echo "=== installing stegseek ==="
wget -q https://github.com/RickdeJager/stegseek/releases/download/v0.6/stegseek_0.6-1.deb
apt-get install -y ./stegseek_0.6-1.deb &> /dev/null
rm -f ./stegseek_0.6-1.deb

#stegoveritas + dependencies
echo "installing stegoveritas"
pip install --upgrade pip &> /dev/null
pip install stegoveritas &> /dev/null
#note: stegoveritas_install_deps auto-downloads underlying tools like zsteg, exam, etc.
stegoveritas_install_deps &> /dev/null

echo "all tools installed successfully"

In [ ]:
import os
import random
import shutil
from pathlib import Path

# define the paths that'll be pooled together
alaska_dir=Path("/kaggle/input/competitions/alaska2-image-steganalysis")
pool_dir=[
    alaska_dir/"JMiPOD",
    alaska_dir/"JUNIWARD",
    alaska_dir/"UERD"
]

sample_dir=Path("/kaggle/working/selected_images")
total= 50

# if imageset_dir alr exists, it won't be made again
sample_dir.mkdir(parents=True, exist_ok=True)

existing_images=list(sample_dir.glob("*.jpg"))

if len(existing_images) >= total:
    print(f"Directory already contains {len(existing_images)} images. Skipping copy.")
else:
    # pool images
    all_images = []
    for folder in pool_dir:
        # use rglob or lower/upper checks if extensions vary
        all_images.extend(list(folder.glob("*.jpg")))
        all_images.extend(list(folder.glob("*.JPG")))
    
    print(f"Total images found in population: {len(all_images)}")
    
    if len(all_images) == 0:
        raise ValueError(
            "No images were found! Check that the ALASKA2 dataset is added to your Kaggle Notebook inputs."
        )
    
    # safely sample 50 images
    sample_size = min(50, len(all_images))
    selected_images = random.sample(all_images, sample_size)
    
    print(f"Successfully sampled {len(selected_images)} images.")
    all_images=[]
    for folder in pool_dir:
        all_images.extend(list(folder.glob("*.jpg")))
    
    # copy them over to image_set.dir
    for src_path in selected_images:
        shutil.copy(src_path, sample_dir / src_path.name)

    # list the 50 images that were randomly selected
    for i, image_path in enumerate(sample_dir.glob("*.jpg"), start=1):
        folder_name=image_path.parent.name
        print(f"{i}. {folder_name}/{image_path.name}")
        
    print(f"Randomly selected and copied {total} images from {len(pool_dir)} folders to {sample_dir}")

In [ ]:
import os
import random
import shutil
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# directory Setup
sample_dir = Path("/kaggle/working/selected_images")
report_dir = Path("/kaggle/working/forensics_reports")
carve_dir = Path("/kaggle/working/extracted_artifacts")

report_dir.mkdir(parents=True, exist_ok=True)
carve_dir.mkdir(parents=True, exist_ok=True)

wordlist_path = '/usr/share/dict/words' 

# 2. Retrieve persistent images and shuffle order per trial
image_paths = list(sample_dir.glob("*.jpg"))

# Change trial seed per trial to change processing order across runs
order_seed = 1  
random.seed(order_seed)
random.shuffle(image_paths)

print(f"{len(image_paths)} images have been loaded. Executing trial order with seed {order_seed}.")

stats_records = []

# run the toolkit
for index, img_path in enumerate(image_paths, 1):
    img_name = img_path.name
    
    # infer category (checks prefix or filename structure)
    category = "Unknown"
    for cat in ['Cover', 'JMiPOD', 'JUNIWARD', 'UERD']:
        if cat.lower() in img_name.lower():
            category = cat
            break

    # telemetry
    byte_size = img_path.stat().st_size
    
    # counters
    binwalk_hits = 0
    foremost_extracted_files = 0
    jsteg_anomaly = 0
    stegseek_cracked = 0

    print(f"[{index}/{len(image_paths)}] Processing {img_name} (Category: {category})...")

    # run binwalk
    bw_res = subprocess.run(['binwalk', str(img_path)], capture_output=True, text=True)
    if bw_res.stdout:
        lines = [l for l in bw_res.stdout.split('\n') if l.strip()]
        if len(lines) > 3:
            binwalk_hits = len(lines) - 3

    # run foremost
    img_carve_out = carve_dir / f"{img_name}_carved"
    subprocess.run(['foremost', '-i', str(img_path), '-o', str(img_carve_out)], capture_output=True)
    if img_carve_out.exists():
        carved_items = [f for f in os.listdir(img_carve_out) if f != 'audit.txt']
        foremost_extracted_files = len(carved_items)

    # run jsteg
    jsteg_out_txt = carve_dir / f"{img_name}_jsteg.txt"
    js_res = subprocess.run(['jsteg', 'reveal', str(img_path), str(jsteg_out_txt)], capture_output=True, text=True)
    if jsteg_out_txt.exists() and jsteg_out_txt.stat().st_size > 0:
        jsteg_anomaly = 1

    # run stegseek
    if os.path.exists(wordlist_path):
        ss_res = subprocess.run(['stegseek', '--wordlist', wordlist_path, str(img_path)], capture_output=True, text=True)
        if "Found passphrase" in ss_res.stderr or "Cracked" in ss_res.stdout:
            stegseek_cracked = 1

    # run stegoveritas
    sv_out = carve_dir / f"{img_name}_veritas"
    subprocess.run(['stegoveritas', str(img_path), '-out', str(sv_out)], capture_output=True)

    # log metrics (recording execution rank/order)
    stats_records.append({
        "trial_execution_order": index,
        "filename": img_name,
        "class": category,
        "group": "Cover" if category == "Cover" else "Stego",
        "file_size_bytes": byte_size,
        "binwalk_hits": binwalk_hits,
        "carved_files_count": foremost_extracted_files,
        "jsteg_anomaly": jsteg_anomaly,
        "stegseek_success": stegseek_cracked
    })

# save structured csv
df = pd.DataFrame(stats_records)
csv_output_path = report_dir / f"forensic_statistical_matrix_order_seed_{order_seed}.csv"
df.to_csv(csv_output_path, index=False)

print(f"Forensic processing complete. Data matrix exported to: {csv_output_path}")